# 02 · Models & evaluation — Stages 4–6
### Predicting impending failures in OpenRCA — **Bank** and **Telecom**

Run **`01_data_pipeline.ipynb`** first (or just run this — importing `pipeline` builds
and caches Stage 1 automatically).

| Stage | Here |
|---|---|
| 1–3 | ← notebook 01 (imported from `pipeline.py`, nothing is recomputed) |
| **5** | Defined first: how a model is scored — row level *and* event level, with bootstrap CIs |
| **4** | Seven model families × six horizons: prior, time-of-day control, logistic, GBM, 1D-CNN, LSTM, BiLSTM+attention |
| **6** | Ablations: look-back, feature width, modality |

**Dataset:** set `DATASET` in the next cell (`"bank"` or `"telecom"`) and re-run. Results
are written to `results/<dataset>/`, so the two never overwrite each other, and the last
section compares whichever ones you have run.

**Preset:** set `OPENRCA_PRESET` before importing if you want something other than
`standard`: `smoke` (~1 min, proves the plumbing), `standard` (the one to report), `full`
(adds day-grouped CV and ablations — long).

In [ ]:
import os
# os.environ["OPENRCA_PRESET"] = "smoke"     # uncomment for a fast dry run

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))   # so `pipeline` resolves next to the notebook

import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from pipeline import *
apply_plot_style()

DATASET = "bank"                   # <-- "bank" or "telecom"
A = load_into(globals(), DATASET)  # injects F, GRID, ONSETS, MASKS, RESULTS, FIGS, ...

print(summary())
print()
print(dataset_overview().to_string())
if not HAS_TORCH:
    print("\n!! torch missing -- the CNN/LSTM/BiLSTM arm will be skipped. pip install torch")

## Stage 5 (defined first) — how we score a model

Accuracy is meaningless at a 1 % base rate. Two families of metric:

**Row level.** *PR-AUC is primary*, always reported next to the base rate it has to
beat — a lift of 1.0 means the model is worthless no matter how pretty the number.
ROC-AUC is reported for comparability but flatters under imbalance.
Precision / recall / F1 use a threshold picked on the **validation** fold.

**Event level** — what an on-call engineer actually cares about. These use a
*fixed false-alarm budget* threshold (30 alarm-minutes/day, chosen on validation)
rather than the F1-optimal one, which at a 1–5 % base rate often collapses to
"never alarm" and reports 0 recall for a model that ranks perfectly well. The budget
threshold is a quantile of the score distribution and uses **no labels**:

* **event recall @ H** — share of faults with at least one alarm in the H minutes before onset;
* **false alarms per day** — raw alarm-minutes *and* de-duplicated alarm episodes;
* **lead time** — how early the first true alarm fires.

**Uncertainty.** With 136 events, ranking models by a third decimal place is
self-deception. Event recall gets a bootstrap CI over events; PR-AUC gets a *block*
bootstrap (60-minute blocks), so correlated neighbouring minutes aren't treated as
independent draws.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

def pick_threshold(y, s):
    """Threshold maximising F1 on the VALIDATION fold. Never on test."""
    if y.sum() == 0 or len(np.unique(s)) < 2:
        return float(np.quantile(s, 0.99))
    p, r, t = precision_recall_curve(y, s)
    f1 = np.where((p[:-1] + r[:-1]) > 0,
                  2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-12), 0)
    return float(t[int(np.argmax(f1))])

ALARM_BUDGET = 30        # alarm-minutes per day we are willing to pay for

def pick_threshold_budget(scores, budget=ALARM_BUDGET):
    """Threshold that spends a fixed alarm budget: `budget` alarm-minutes per day.

    The F1-optimal threshold is unstable at a 1-5 % base rate -- it can collapse to
    "never alarm" and report 0 event recall for a model that ranks perfectly well,
    and it does not transfer across folds because the score distributions shift.
    A fixed alarm budget makes event recall comparable across models and mirrors how
    an alerting rule is really tuned ("I will tolerate N pages a day").

    This is a quantile of the score distribution and uses **no labels**, so applying
    it to the test fold is not leakage -- but say so in the write-up, and always
    report the alarm load beside the recall it bought.
    """
    q = 1.0 - min(max(budget / DAY_MINUTES, 1e-4), 0.5)
    return float(np.quantile(scores, q))

def row_metrics(y, s, thr):
    base = float(y.mean())
    out = dict(base_rate=base, pr_auc=np.nan, pr_lift=np.nan, roc_auc=np.nan)
    if 0 < y.sum() < len(y):
        out["pr_auc"]  = float(average_precision_score(y, s))
        out["pr_lift"] = out["pr_auc"] / base if base > 0 else np.nan
        out["roc_auc"] = float(roc_auc_score(y, s))
    pred = s >= thr
    tp = float((pred & (y == 1)).sum()); fp = float((pred & (y == 0)).sum())
    fn = float((~pred & (y == 1)).sum())
    out["precision"] = tp / (tp + fp) if tp + fp else 0.0
    out["recall"]    = tp / (tp + fn) if tp + fn else 0.0
    den = out["precision"] + out["recall"]
    out["f1"] = 2 * out["precision"] * out["recall"] / den if den else 0.0
    return out

def event_metrics(fold, h, rows_test, scores, thr, y_test, n_boot=0):
    """Per-fault detection, false-alarm load, and lead time."""
    score_of = dict(zip(rows_test.tolist(), scores.tolist()))
    ev = ONSETS[ONSETS["datetime"].str[:10].isin(fold.test)]
    detected, leads, reasons = [], [], []
    for _, r in ev.iterrows():
        i = ROW_OF[int(r["onset"])]
        hit = None
        for lead in range(h, 0, -1):            # earliest minute in the window first
            j = i - lead
            if j in score_of and score_of[j] >= thr:
                hit = lead; break
        detected.append(hit is not None); reasons.append(r["reason"])
        if hit is not None:
            leads.append(hit)
    detected = np.array(detected, dtype=bool)

    alarm = scores >= thr
    fa_minutes, fa_runs = int((alarm & (y_test == 0)).sum()), 0
    prev_row, prev_fa = None, False
    for row, a, yy in zip(rows_test.tolist(), alarm.tolist(), y_test.tolist()):
        is_fa = bool(a and yy == 0)
        if is_fa and not (prev_fa and prev_row == row - 1):
            fa_runs += 1
        prev_fa, prev_row = is_fa, row
    n_days = max(len(fold.test), 1)

    out = dict(n_events=int(len(detected)),
               event_recall=float(detected.mean()) if len(detected) else np.nan,
               fa_minutes_per_day=fa_minutes / n_days,
               fa_episodes_per_day=fa_runs / n_days,
               median_lead_min=float(np.median(leads)) if leads else np.nan)
    if n_boot and len(detected):
        rng = np.random.default_rng(SEED)
        bs = [detected[rng.integers(0, len(detected), len(detected))].mean()
              for _ in range(n_boot)]
        out["event_recall_lo"], out["event_recall_hi"] = np.percentile(bs, [2.5, 97.5])
    out["_detected"], out["_reasons"] = detected, reasons
    return out

def block_bootstrap_prauc(y, s, n_boot=300, block=60):
    """PR-AUC CI that respects minute-to-minute autocorrelation."""
    if not (0 < y.sum() < len(y)) or n_boot == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(SEED)
    starts = np.arange(0, max(len(y) - block, 1))
    n_blocks = max(len(y) // block, 1)
    vals = []
    for _ in range(n_boot):
        idx = np.concatenate([np.arange(p, min(p + block, len(y)))
                              for p in rng.choice(starts, n_blocks, replace=True)])
        if 0 < y[idx].sum() < len(idx):
            vals.append(average_precision_score(y[idx], s[idx]))
    return tuple(np.percentile(vals, [2.5, 97.5])) if vals else (np.nan, np.nan)

print("evaluation helpers ready")

## Stage 4 — the model zoo

Seven families, all fed identical windows, all on the same tuning budget. Two of them
exist purely to keep the others honest:

* **`prior`** — predicts the training base rate. PR-AUC at or below this means the
  model learned nothing, whatever its ROC-AUC says.
* **`time_of_day`** — sees *only* `sin/cos` of the minute of day. If this scores near
  the LSTM at H=30, that horizon is detecting the diurnal load rhythm rather than
  impending failure. This control is the difference between a finding and an artefact.

The sequence models (CNN, LSTM, BiLSTM+attention) see the raw `(60, k)` window. The
tabular models (logistic, GBM) see the same window flattened to six summary statistics
per feature — mean, std, min, max, last, slope — so the comparison is about *temporal
structure*, not about who got more information.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

def m_prior(d):
    p = float(d["y_train"].mean())
    return np.full(len(d["y_val"]), p), np.full(len(d["y_test"]), p)

def _tod(rows):
    m = POS_IN_DAY[rows] / DAY_MINUTES * 2 * np.pi
    return np.c_[np.sin(m), np.cos(m), np.sin(2 * m), np.cos(2 * m)]

def m_time_of_day(d):
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(_tod(d["rows_train"]), d["y_train"])
    return (clf.predict_proba(_tod(d["rows_val"]))[:, 1],
            clf.predict_proba(_tod(d["rows_test"]))[:, 1])

def m_logistic(d):
    Xtr, Xva, Xte = (window_stats(d[f"X_{s}"]) for s in ("train", "val", "test"))
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=0.1)
    clf.fit(Xtr, d["y_train"])
    return clf.predict_proba(Xva)[:, 1], clf.predict_proba(Xte)[:, 1]

def m_gbm(d):
    Xtr, Xva, Xte = (window_stats(d[f"X_{s}"]) for s in ("train", "val", "test"))
    y = d["y_train"]
    if HAS_XGB:
        import xgboost as xgb
        clf = xgb.XGBClassifier(
            n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
            colsample_bytree=0.6, eval_metric="aucpr", tree_method="hist",
            scale_pos_weight=float((y == 0).sum() / max((y == 1).sum(), 1)),
            n_jobs=4, random_state=SEED)
    else:
        clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06,
                                             max_depth=5, class_weight="balanced",
                                             random_state=SEED)
    clf.fit(Xtr, y)
    return clf.predict_proba(Xva)[:, 1], clf.predict_proba(Xte)[:, 1]

CLASSICAL = {"prior": m_prior, "time_of_day": m_time_of_day,
             "logistic": m_logistic, "gbm": m_gbm}
print("classical:", list(CLASSICAL), "| gbm backend:",
      "xgboost" if HAS_XGB else "sklearn HistGradientBoosting")

In [ ]:
if HAS_TORCH:
    import torch, torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader

    class CNN1D(nn.Module):
        """Local shape detection without recurrence -- the fair neural baseline."""
        def __init__(self, f, ch=64):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv1d(f, ch, 5, padding=2), nn.ReLU(), nn.BatchNorm1d(ch),
                nn.Conv1d(ch, ch, 3, padding=1), nn.ReLU(), nn.AdaptiveMaxPool1d(1))
            self.head = nn.Linear(ch, 1)
        def forward(self, x):                                  # x: (n, L, f)
            return self.head(self.net(x.transpose(1, 2)).squeeze(-1)).squeeze(-1)

    class LSTMNet(nn.Module):
        """The hypothesis under test."""
        def __init__(self, f, hidden=64, layers=1, p=0.2):
            super().__init__()
            self.rnn = nn.LSTM(f, hidden, layers, batch_first=True,
                               dropout=p if layers > 1 else 0.0)
            self.drop, self.head = nn.Dropout(p), nn.Linear(hidden, 1)
        def forward(self, x):
            out, _ = self.rnn(x)
            return self.head(self.drop(out[:, -1, :])).squeeze(-1)

    class BiLSTMAttn(nn.Module):
        """Does context in both directions plus learned focus beat a plain LSTM?"""
        def __init__(self, f, hidden=64, p=0.2):
            super().__init__()
            self.rnn = nn.LSTM(f, hidden, 1, batch_first=True, bidirectional=True)
            self.att = nn.Linear(2 * hidden, 1)
            self.drop, self.head = nn.Dropout(p), nn.Linear(2 * hidden, 1)
        def forward(self, x):
            out, _ = self.rnn(x)                               # (n, L, 2h)
            w = torch.softmax(self.att(out).squeeze(-1), dim=1)
            return self.head(self.drop((out * w.unsqueeze(-1)).sum(1))).squeeze(-1)

    def _predict(model, X, bs=512):
        model.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(X), bs):
                out.append(torch.sigmoid(model(torch.from_numpy(X[i:i + bs]))).numpy())
        return np.concatenate(out) if out else np.zeros(0)

    def train_torch(make, d, epochs=None, lr=1e-3, bs=128, patience=4):
        """Pos-weighted BCE, gradient clipping, early stop on validation PR-AUC."""
        epochs = epochs or CFG["epochs"]
        torch.manual_seed(SEED)
        Xtr, ytr = d["X_train"], d["y_train"]
        model = make(Xtr.shape[2])
        pw = torch.tensor([max((ytr == 0).sum() / max((ytr == 1).sum(), 1), 1.0)],
                          dtype=torch.float32)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
        dl = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)),
                        batch_size=bs, shuffle=True)
        best, best_state, bad = -1.0, None, 0
        for _ in range(epochs):
            model.train()
            for xb, yb in dl:
                opt.zero_grad()
                loss_fn(model(xb), yb).backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step()
            sv = _predict(model, d["X_val"])
            score = (average_precision_score(d["y_val"], sv)
                     if 0 < d["y_val"].sum() < len(d["y_val"]) else 0.0)
            if score > best:
                best, bad = score, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    break
        if best_state:
            model.load_state_dict(best_state)
        return _predict(model, d["X_val"]), _predict(model, d["X_test"])

    NEURAL = {"cnn1d":       lambda d: train_torch(CNN1D, d),
              "lstm":        lambda d: train_torch(LSTMNet, d),
              "bilstm_attn": lambda d: train_torch(BiLSTMAttn, d)}
else:
    NEURAL = {}

MODELS = {**CLASSICAL, **NEURAL}
print("all models:", list(MODELS))

## The experiment runner

For every (fold × horizon × model): fit on train, pick the alarm threshold on
validation, score test once. Out-of-fold test scores are kept so we can draw PR curves
and slice event recall by fault reason afterwards.

In [ ]:
def run_experiments(folds, horizons, models=None, boot=None, verbose=True):
    models = models or MODELS
    boot = CFG["boot"] if boot is None else boot
    rows, preds = [], {}
    for fold in folds:
        for h in horizons:
            d = fold_data(fold, h)
            for name, fn in models.items():
                t0 = time.time()
                try:
                    sv, st = fn(d)
                except Exception as e:
                    print(f"  !! {name} h{h} {fold.name}: {type(e).__name__}: {e}")
                    continue
                thr   = pick_threshold(d["y_val"], sv)          # row metrics
                thr_b = pick_threshold_budget(st)              # event metrics, label-free
                r = dict(split=fold.name, horizon=h, model=name, threshold_f1=thr,
                         threshold_budget=thr_b, seconds=round(time.time() - t0, 1))
                r.update(row_metrics(d["y_test"], st, thr))
                lo, hi = block_bootstrap_prauc(d["y_test"], st, n_boot=min(boot, 300))
                r["pr_auc_lo"], r["pr_auc_hi"] = lo, hi
                em = event_metrics(fold, h, d["rows_test"], st, thr_b, d["y_test"], boot)
                preds[(fold.name, h, name)] = dict(
                    rows=d["rows_test"], scores=st, y=d["y_test"], thr=thr_b,
                    fold=fold,
                    detected=em.pop("_detected"), reasons=em.pop("_reasons"))
                r.update(em)
                rows.append(r)
                if verbose:
                    print(f"  {fold.name:7s} h{h:<3d} {name:12s} "
                          f"PR-AUC {r['pr_auc']:.3f} (lift {r['pr_lift']:5.2f}x)  "
                          f"event recall {r['event_recall']:.2f}  "
                          f"FA/day {r['fa_episodes_per_day']:5.1f}  {r['seconds']:5.1f}s")
            del d
    return pd.DataFrame(rows), preds

SELECTED = DATASET                 # remember it: load_into() rebinds DATASET each time
DATASETS_TO_RUN = [SELECTED]       # set to ["bank", "telecom"] to do both in one go

ALL_RES, ALL_PREDS = {}, {}
for _name in DATASETS_TO_RUN:
    _A = load_into(globals(), _name)
    print(f"\n=== {_name} : {len(DAYS)} days x {DAY_MINUTES} min, {len(ONSETS)} events ===")
    t0 = time.time()
    _res, _preds = run_experiments(build_folds(), CFG["horizons"])
    _res.insert(0, "dataset", _name)
    _res.to_csv(RESULTS / "results.csv", index=False)
    ALL_RES[_name], ALL_PREDS[_name] = _res, _preds
    print(f"  {_name}: {time.time() - t0:.0f}s, {len(_res)} model runs "
          f"-> {RESULTS / 'results.csv'}")

# restore the dataset selected at the top; the analysis cells below describe it
A = load_into(globals(), SELECTED)
RES, PREDS = ALL_RES[SELECTED], ALL_PREDS[SELECTED]
FOLDS = build_folds()
print(f"\nanalysis cells below describe: {DATASET}")

## Results

Read the table in this order: **PR-AUC lift** first (is it beating the base rate at
all?), then **event recall** and **false alarms per day** together (recall bought with
200 alarms a day is not recall), then the **time-of-day control** row — if it is close
to the sequence models at a horizon, that horizon's result is a rhythm detector.

In [ ]:
pd.set_option("display.width", 170, "display.max_columns", 40)

def _fmt(df, f):
    return df.map(f) if hasattr(df, "map") and pd.__version__ >= "2.1" else df.applymap(f)

order = [m for m in MODELS if m in set(RES["model"])]
def pivot(metric, fmt="{:.3f}"):
    p = (RES.pivot_table(index="model", columns="horizon", values=metric, aggfunc="mean")
            .reindex(order))
    return p.pipe(_fmt, lambda v: fmt.format(v) if pd.notna(v) else "—")

print("PR-AUC (test)"); print(pivot("pr_auc").to_string())
print("\nPR-AUC lift over base rate  (1.00 = worthless)")
print(pivot("pr_lift", "{:.2f}x").to_string())
print("\nEvent recall @ H"); print(pivot("event_recall", "{:.2f}").to_string())
print("\nFalse-alarm episodes per day (at the 30 alarm-min/day budget)")
print(pivot("fa_episodes_per_day", "{:.1f}").to_string())
print("\nMedian lead time of first true alarm (min)")
print(pivot("median_lead_min", "{:.0f}").to_string())

best = (RES[~RES["model"].isin(["prior", "time_of_day"])]
        .sort_values("pr_auc", ascending=False)
        .groupby("horizon").head(1)
        .loc[:, ["horizon", "model", "pr_auc", "pr_lift", "event_recall",
                 "fa_episodes_per_day", "median_lead_min"]])
print("\nBest model per horizon:"); print(best.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))

ax = axes[0]
for k, m in enumerate(order):
    sub = RES[RES["model"] == m].groupby("horizon")["pr_lift"].mean()
    if sub.empty: continue
    ax.plot(sub.index, sub.values, "-o", color=PAL[k % len(PAL)], markersize=5, label=m)
    ax.annotate(m, (sub.index[-1], sub.values[-1]), textcoords="offset points",
                xytext=(6, 0), fontsize=8, color=PAL[k % len(PAL)], va="center")
ax.axhline(1.0, color=INK, lw=1.2, ls="--")
ax.annotate("base rate (worthless)", (HORIZONS[0], 1.0), textcoords="offset points",
            xytext=(0, 6), fontsize=8, color=MUTED)
ax.set_title("PR-AUC lift over the base rate"); ax.set_xlabel("horizon H (minutes)")
ax.set_ylabel("PR-AUC / base rate"); ax.set_xlim(0, max(CFG["horizons"]) * 1.25)

ax = axes[1]
for k, m in enumerate(order):
    sub = RES[RES["model"] == m].groupby("horizon")[["event_recall", "fa_episodes_per_day"]].mean()
    if sub.empty: continue
    ax.plot(sub["fa_episodes_per_day"], sub["event_recall"], "o", color=PAL[k % len(PAL)],
            markersize=7, label=m)
ax.set_title("The operating trade-off (each point = one model × horizon)")
ax.set_xlabel("false-alarm episodes per day"); ax.set_ylabel("event recall @ H")
ax.set_ylim(-0.03, 1.03); ax.legend(ncol=2, loc="lower right")

fig.tight_layout(); fig.savefig(FIGS / "04_results.png", bbox_inches="tight")
plt.show()

In [ ]:
# Which fault reasons are predictable at all? Often a more interesting finding
# than the headline AUC -- CPU/memory faults announce themselves, packet loss does not.
H_FOCUS = 5 if 5 in CFG["horizons"] else CFG["horizons"][0]
cand = [m for m in order if m not in ("prior", "time_of_day")]
pick = (RES[(RES["horizon"] == H_FOCUS) & (RES["model"].isin(cand))]
        .sort_values("pr_auc", ascending=False).head(1))

if len(pick):
    key = (pick.iloc[0]["split"], H_FOCUS, pick.iloc[0]["model"])
    p = PREDS[key]
    by_reason = (pd.DataFrame({"reason": p["reasons"], "detected": p["detected"]})
                 .groupby("reason")["detected"].agg(["mean", "size"])
                 .sort_values("mean"))
    fig, ax = plt.subplots(figsize=(9.5, 0.42 * len(by_reason) + 1.6))
    ax.barh(range(len(by_reason)), by_reason["mean"].values, color=PAL[0], height=0.62)
    ax.set_yticks(range(len(by_reason)))
    ax.set_yticklabels([f"{r}  (n={n})" for r, n in
                        zip(by_reason.index, by_reason["size"])], fontsize=8.5)
    for i, v in enumerate(by_reason["mean"].values):
        ax.text(v + 0.015, i, f"{v:.0%}", va="center", fontsize=8, color=MUTED)
    ax.set_xlim(0, 1.12); ax.grid(axis="y", visible=False)
    ax.set_title(f"Event recall by fault reason — {key[2]}, H={H_FOCUS} min, {key[0]} split")
    ax.set_xlabel("share of faults caught")
    fig.tight_layout(); fig.savefig(FIGS / "05_recall_by_reason.png", bbox_inches="tight")
    plt.show()
    print(by_reason.round(3).to_string())

In [ ]:
# The operating curve an SRE would actually be handed: sweep the alarm budget and
# watch event recall trade against the false-alarm load it costs.
BUDGETS = [5, 10, 20, 40, 80, 160, 320]
top3 = (RES[(RES["horizon"] == H_FOCUS) & (~RES["model"].isin(["prior"]))]
        .sort_values("pr_auc", ascending=False)["model"].head(3).tolist())

fig, ax = plt.subplots(figsize=(7.2, 4.8))
for k, m in enumerate(top3):
    key = next((kk for kk in PREDS if kk[1] == H_FOCUS and kk[2] == m), None)
    if key is None: continue
    p = PREDS[key]
    xs, ys = [], []
    for b in BUDGETS:
        thr = float(np.quantile(p["scores"], 1.0 - min(b / DAY_MINUTES, 0.5)))
        em = event_metrics(p["fold"], H_FOCUS, p["rows"], p["scores"], thr, p["y"])
        xs.append(em["fa_episodes_per_day"]); ys.append(em["event_recall"])
    ax.plot(xs, ys, "-o", color=PAL[k], markersize=5, label=m)
    ax.annotate(m, (xs[-1], ys[-1]), textcoords="offset points", xytext=(6, 0),
                fontsize=8, color=PAL[k], va="center")
ax.set_title(f"Event recall vs alarm load — H = {H_FOCUS} min")
ax.set_xlabel("false-alarm episodes per day"); ax.set_ylabel("event recall @ H")
ax.set_ylim(-0.03, 1.03)
fig.tight_layout(); fig.savefig(FIGS / "07_operating_curve.png", bbox_inches="tight")
plt.show()

In [ ]:
# Precision-recall curves at the focus horizon. PR, not ROC -- ROC hides the imbalance.
fig, ax = plt.subplots(figsize=(6.4, 5.0))
shown = 0
for k, m in enumerate(order):
    key = next((kk for kk in PREDS if kk[1] == H_FOCUS and kk[2] == m), None)
    if key is None: continue
    p = PREDS[key]
    if not (0 < p["y"].sum() < len(p["y"])): continue
    pr, rc, _ = precision_recall_curve(p["y"], p["scores"])
    ax.plot(rc, pr, color=PAL[k % len(PAL)], label=m); shown += 1
base = PREDS[next(iter(k for k in PREDS if k[1] == H_FOCUS))]["y"].mean()
ax.axhline(base, color=INK, ls="--", lw=1.2)
ax.annotate(f"base rate {base:.1%}", (0.02, base), textcoords="offset points",
            xytext=(0, 6), fontsize=8, color=MUTED)
ax.set_title(f"Precision–recall, H = {H_FOCUS} min")
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
if shown >= 2: ax.legend(loc="upper right")
fig.tight_layout(); fig.savefig(FIGS / "06_pr_curves.png", bbox_inches="tight")
plt.show()

## Stage 6 — ablations

Three questions the committee will ask. Each re-runs the LSTM (and the GBM as a
reference) at the focus horizon with one thing changed.

1. **Does temporal depth help?** Look-back ∈ {15, 30, 60, 120} minutes.
2. **How much feature width do we need?** top-k ∈ {32, 128, 384}.
3. **Which modality carries the signal?** container KPIs only vs app metrics only vs both.

In [ ]:
ABL_MODELS = {k: v for k, v in MODELS.items() if k in ("gbm", "lstm")}

def ablate(label, **kw):
    out = []
    for name, fn in ABL_MODELS.items():
        d = fold_data(FOLDS[0], H_FOCUS, **kw)
        sv, st = fn(d)
        thr = pick_threshold(d["y_val"], sv)
        r = dict(ablation=label, model=name)
        r.update(row_metrics(d["y_test"], st, thr))
        r.update({k: v for k, v in
                  event_metrics(FOLDS[0], H_FOCUS, d["rows_test"], st, thr,
                                d["y_test"]).items() if not k.startswith("_")})
        out.append(r); del d
    return out

ABL = []
if CFG["ablations"]:
    for L in (15, 30, 60, 120):
        ABL += ablate(f"lookback={L}", lookback=L)
    for k in (32, 128, 384):
        ABL += ablate(f"top_k={k}", k=k)
    app_cols = np.flatnonzero(COMPONENT_OF == "app")
    con_cols = np.flatnonzero(COMPONENT_OF != "app")
    ABL += ablate("modality=container", restrict=con_cols)
    ABL += ablate("modality=app",       restrict=app_cols)
    ABL += ablate("modality=both",      restrict=None)

    ABL = pd.DataFrame(ABL)
    ABL.to_csv(RESULTS / "ablations.csv", index=False)
    print(f"Ablations at H={H_FOCUS}, {FOLDS[0].name} split\n")
    print(ABL.pivot_table(index="ablation", columns="model", values="pr_auc")
             .pipe(_fmt, "{:.3f}".format).to_string())
    print("\nEvent recall")
    print(ABL.pivot_table(index="ablation", columns="model", values="event_recall")
             .pipe(_fmt, "{:.2f}".format).to_string())
else:
    ABL = pd.DataFrame()
    print("ablations disabled for this preset (set PRESET='standard' or 'full')")

## Bank vs Telecom

Whichever datasets you have run are compared here from `results/<dataset>/results.csv` —
so you can run Bank now, Telecom later, and still get the comparison. Both went through
identical Stage 1–3 code, so differences are about the systems, not the preprocessing.

Read it with the sample sizes in mind: Telecom has 51 events to Bank's 136, so its
confidence intervals are roughly 1.6× wider. A model that looks better on Telecom by
0.05 PR-AUC is not better.

In [ ]:
def load_results(name):
    f = HERE / "results" / name / "results.csv"
    if not f.exists():
        return None
    r = pd.read_csv(f)
    if "dataset" not in r.columns:
        r.insert(0, "dataset", name)
    return r

COMP = pd.concat([r for r in (load_results(n) for n in DATASETS) if r is not None],
                 ignore_index=True) if True else None

if COMP is None or COMP["dataset"].nunique() < 2:
    have = sorted(COMP["dataset"].unique()) if COMP is not None else []
    print("Only these datasets have results so far:", have or "none")
    print('Set DATASETS_TO_RUN = ["bank", "telecom"] in the runner cell, or switch')
    print("DATASET at the top and re-run, then come back to this cell.")
else:
    ov = dataset_overview()
    print(ov.to_string())

    piv = (COMP[~COMP["model"].isin(["prior"])]
           .pivot_table(index=["model"], columns=["dataset", "horizon"], values="pr_lift"))
    print("\nPR-AUC lift over base rate")
    print(piv.round(2).to_string())

    best = (COMP[~COMP["model"].isin(["prior", "time_of_day"])]
            .sort_values("pr_auc", ascending=False)
            .groupby(["dataset", "horizon"]).head(1)
            .sort_values(["dataset", "horizon"])
            .loc[:, ["dataset", "horizon", "model", "pr_auc", "pr_lift",
                     "event_recall", "fa_episodes_per_day", "n_events"]])
    print("\nBest model per dataset x horizon")
    print(best.to_string(index=False))

In [ ]:
if COMP is not None and COMP["dataset"].nunique() >= 2:
    names = sorted(COMP["dataset"].unique())
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))

    # (a) best achievable lift per horizon, per dataset
    ax = axes[0]
    for k, n in enumerate(names):
        sub = (COMP[(COMP["dataset"] == n) & (~COMP["model"].isin(["prior", "time_of_day"]))]
               .groupby("horizon")["pr_lift"].max())
        ax.plot(sub.index, sub.values, "-o", markersize=5,
                color=DATASET_COLOR.get(n, PAL[k]), label=n)
        ax.annotate(n, (sub.index[-1], sub.values[-1]), textcoords="offset points",
                    xytext=(6, 0), fontsize=8, color=DATASET_COLOR.get(n, PAL[k]), va="center")
    ax.axhline(1.0, color=INK, ls="--", lw=1.2)
    ax.set_title("Best PR-AUC lift achieved, by horizon")
    ax.set_xlabel("horizon H (min)"); ax.set_ylabel("PR-AUC / base rate")
    ax.set_xlim(0, max(COMP["horizon"]) * 1.3)

    # (b) does the sequence model beat the tabular one? per dataset, per horizon
    ax = axes[1]
    for k, n in enumerate(names):
        sub = COMP[COMP["dataset"] == n].pivot_table(index="horizon", columns="model",
                                                     values="pr_auc")
        if not {"lstm", "gbm"} <= set(sub.columns):
            continue
        ax.plot(sub.index, sub["lstm"] - sub["gbm"], "-o", markersize=5,
                color=DATASET_COLOR.get(n, PAL[k]), label=n)
    ax.axhline(0, color=INK, ls="--", lw=1.2)
    ax.annotate("LSTM better above the line", (0.03, 0.94), xycoords="axes fraction",
                fontsize=8, color=MUTED)
    ax.set_title("LSTM minus GBM (PR-AUC)")
    ax.set_xlabel("horizon H (min)"); ax.set_ylabel("difference in PR-AUC")
    ax.legend()

    # (c) the control model -- how much is just daily rhythm?
    ax = axes[2]
    w = 0.36
    xs = np.arange(len(names))
    for k, m in enumerate(["time_of_day", "lstm"]):
        vals = [COMP[(COMP["dataset"] == n) & (COMP["model"] == m) &
                     (COMP["horizon"] == max(COMP["horizon"]))]["pr_lift"].mean()
                for n in names]
        ax.bar(xs + (k - 0.5) * w, vals, width=w, color=PAL[k * 4], label=m)
        for x, v in zip(xs + (k - 0.5) * w, vals):
            if pd.notna(v):
                ax.text(x, v + 0.03, f"{v:.2f}x", ha="center", fontsize=8, color=MUTED)
    ax.axhline(1.0, color=INK, ls="--", lw=1.2)
    ax.set_xticks(xs); ax.set_xticklabels(names)
    ax.set_title(f"Rhythm check at H = {max(COMP['horizon'])} min")
    ax.set_ylabel("PR-AUC lift"); ax.legend()

    fig.tight_layout()
    fig.savefig(HERE / "results" / "00_bank_vs_telecom.png", bbox_inches="tight")
    plt.show()

## Reading the results honestly

Four things to check before you write a conclusion:

* **`prior` looks like it has high event recall.** It doesn't. A constant score means
  every minute ties at the budget threshold, so it alarms on everything — read its
  recall next to its false-alarm load, which is several times any real model's. That
  pairing is exactly why both columns are reported.
* **Is `time_of_day` close to your best model at H=30?** If so, that horizon is
  detecting the daily load rhythm, not impending failure. Report the control alongside
  the headline or the headline is not a finding.
* **Do the PR-AUC confidence intervals overlap?** With 136 events, gaps under roughly
  0.05 PR-AUC are noise. `pr_auc_lo` / `pr_auc_hi` are in `results.csv` — use them
  rather than ranking by the mean.
* **Did the GBM beat the LSTM?** On tabular telemetry it very often does. That is a
  legitimate, publishable result, and it is exactly why the classical arm gets the same
  budget as the neural one. Do not quietly tune one harder.
* **Chronological vs CV.** `full` runs both. The chronological number will be worse —
  the 03-23…03-25 block has a different component inventory. That gap *is* a finding
  about concept drift; don't average it away.

* **Cross-system claims need both numbers.** "The LSTM works" is only a finding if it
  holds on Bank *and* Telecom — two systems with different topologies, modalities and
  fault types. If it holds on one and not the other, that difference is the result, and
  the honest write-up leads with it rather than with the better of the two.
* **Telecom's 6-hour days weaken the `time_of_day` control.** It sees only 360 clock
  positions, all in the small hours, so a low score there does *not* clear the
  rhythm-artefact concern the way it does on Bank.

And the standing caveat: these are synthetic injected faults at ~1 % of minutes, in a
ten-day (Bank) or fifteen-half-day (Telecom) window. Nothing here generalises to
production incident prediction without saying so.

In [ ]:
for _name, _res in ALL_RES.items():
    _res.to_csv(HERE / "results" / _name / "results.csv", index=False)
if len(ABL):
    ABL.to_csv(RESULTS / "ablations.csv", index=False)
print("saved under results/:")
for p in sorted((HERE / "results").rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(HERE))